### Исследование результатов работы baseline-моделей:

- Linear Regression
- Elastic Net

In [27]:
import pandas as pd
import sys
from pathlib import Path
from numpy import ndarray, arange
# чтобы убрать ошибку vs code
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src.preprocessing import LinearDataPreprocessor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../data/train.csv')
preprocessor = LinearDataPreprocessor(drop='first')

data_train, data_test = train_test_split(df, test_size=0.2, random_state=42)

data_train = preprocessor.clean_train_data(data_train)

data_train_processed = preprocessor.fit_transform(data_train)
data_test_processed = preprocessor.transform(data_test)


y_train = data_train_processed[preprocessor.target_column]
X_train = data_train_processed.drop(columns=preprocessor.target_column)

y_test = data_test_processed[preprocessor.target_column]
X_test = data_test_processed.drop(columns=preprocessor.target_column)

c:\Users\Huawei\python_programming\ml_pipeline\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


### Подготовка функции для создания отчетов об метриках на выборках моделей:
- mae
- rmse
- mape
- r2

In [18]:
def count_metrics(y_true:ndarray|pd.Series, y_pred:ndarray)->dict:
    mae = mean_absolute_error(y_true,y_pred)
    rmse = root_mean_squared_error(y_true,y_pred)
    mape = mean_absolute_percentage_error(y_true,y_pred)
    r2 = r2_score(y_true,y_pred)
    return {
        "mae":mae,
        "rmse":rmse,
        "mape":mape,
        "r2":r2
    }

### Обучение моделей:

In [19]:
regressor = LinearRegression()
elasticnet = ElasticNet()

regressor.fit(X_train,y_train)
elasticnet.fit(X_train,y_train)

predicts_train_regressor = regressor.predict(X_train)
predicts_test_regressor = regressor.predict(X_test)

predicts_train_elasticnet = elasticnet.predict(X_train)
predicts_test_elasticnet = elasticnet.predict(X_test)

### Оценка метрик

In [20]:
metrics_train_regressor = count_metrics(y_train,predicts_train_regressor)
metrics_test_regressor = count_metrics(y_test,predicts_test_regressor)

print("Метрики линейной регрессии на train:\n ",metrics_train_regressor)
print("Метрики линейной регрессии на test:\n ",metrics_test_regressor)

Метрики линейной регрессии на train:
  {'mae': 20071.290399558166, 'rmse': 28519.714391294798, 'mape': 0.12006367234257062, 'r2': 0.8638562523526726}
Метрики линейной регрессии на test:
  {'mae': 21924.904657961597, 'rmse': 35679.02149767047, 'mape': 0.12882070601434428, 'r2': 0.8340367096984178}


In [24]:
metrics_train_elasticnet = count_metrics(y_train,predicts_train_elasticnet)
metrics_test_elasticnet = count_metrics(y_test,predicts_test_elasticnet)

print("Метрики линейной регрессии с регуляризацией на train:\n ",metrics_train_elasticnet)
print("Метрики линейной регрессии с регуляризацией на test:\n ",metrics_test_elasticnet)

Метрики линейной регрессии с регуляризацией на train:
  {'mae': 20562.935528917544, 'rmse': 31592.795004235613, 'mape': 0.11616563136765433, 'r2': 0.8329357765569028}
Метрики линейной регрессии с регуляризацией на test:
  {'mae': 22633.345514996698, 'rmse': 39485.16383500819, 'mape': 0.12763496263746074, 'r2': 0.796739001580327}


### Добавим кросс валидацию для поиска наилучших гиперпараметров линейной регрессии с регуляризацией:

In [29]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'l1_ratio': arange(start=0.1,stop=1.0,step=0.5)
}
grid_search = GridSearchCV(
    estimator=ElasticNet(max_iter=10000),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error', # Ищем модель с минимальной MAE
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
best_elasticnet = grid_search.best_estimator_

predicts_test_best_en = best_elasticnet.predict(X_test)
predicts_train_best_en = best_elasticnet.predict(X_train)

print("\nМетрики ЛУЧШЕГО ElasticNet на train:\n", count_metrics(y_train, predicts_train_best_en))
print("Метрики ЛУЧШЕГО ElasticNet на test:\n", count_metrics(y_test, predicts_test_best_en))

Лучшие параметры: {'alpha': 0.1, 'l1_ratio': np.float64(0.1)}

Метрики ЛУЧШЕГО ElasticNet на train:
 {'mae': 19921.555494080734, 'rmse': 28935.367423824562, 'mape': 0.11702224211790781, 'r2': 0.859858952218626}
Метрики ЛУЧШЕГО ElasticNet на test:
 {'mae': 21530.61820295698, 'rmse': 36234.434613082, 'mape': 0.12416078555158001, 'r2': 0.8288294113880723}


### Выводы:

- aplpha = 0.1 означает, что модель выбрала слабую регуляризацию, что означает, что все признаки хорошо описывают данные и их не нужно штрафовать
- l1_ratio = 0.1 означает, что L2 сильно доминирует над L1: модель считает, что все фичи полезные, только стоит контролировать веса признаков.


На тестовой выборке средняя ошибка ElasticNet меньше на 0.4% чем у обычной линейной регрессии, поэтому как baseline выбираем ElasticNet

### Сохраняем модель:

In [30]:
import joblib
import os

os.makedirs('../models',exist_ok=True)

joblib.dump(best_elasticnet,'../models/baseline_model.pkl')
joblib.dump(preprocessor,'../models/linear_preprocessor.pkl')

print("Пайплайн успешно сохранен")

Пайплайн успешно сохранен
